In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import warnings
warnings.filterwarnings('ignore')
pd.options.display.max_columns = None

In [ ]:
# Reset cells output
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')
pd.reset_option('display.expand_frame_repr')

In [ ]:
print("hello")

In [ ]:
# Define the path to the processed data based on the current user
current_user = os.getlogin()

# Define output directory (notebook is in ./notebooks; data is ../data)
DATA_DIR     = Path(rf"/home/{current_user}/local-share")
OUT_DIR      = DATA_DIR / "processed"

combined_path = OUT_DIR / "v2_combined.csv"

# Let pandas detect the delimiter
data = pd.read_csv(combined_path, sep=None, engine="python", encoding="utf-8-sig")
print(data.shape)
data


In [ ]:
data.columns

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Identify and PRINT "unnecessary" ID-like columns and CODE-like
# columns from `data` without modifying it. Results are stored in variables for
# the next cell.
# -----------------------------------------------------------------------------
# Heuristics:
# • ID-like: names that are (case-insensitively) 'id', end with '_id'/'_ids',
#   start with 'id_', or contain '_id_' (e.g., 'student_id', 'cohort_ids').
# • CODE-like: names containing 'code' (e.g., 'degree_code', 'code_letters').
# • Always KEEP the primary key 'SINH_ID' (not flagged for removal).
# Outputs:
#   - unnecessary_id_cols   : list[str]
#   - unnecessary_code_cols : list[str]
# -----------------------------------------------------------------------------

cols = list(data.columns)

# 1) Identify ID-like columns (but exclude SINH_ID)
id_patterns = [
    r'^id$', r'^ids$',
    r'.*_id$', r'.*_ids$',
    r'^id_.*', r'^ids_.*',
    r'.*_id_.*', r'.*_ids_.*'
]
id_regex = re.compile('|'.join(id_patterns), flags=re.IGNORECASE)
unnecessary_id_cols = [c for c in cols if id_regex.search(c) and c.upper() != 'SINH_ID']

# 2) Identify CODE-like columns
code_regex = re.compile(r'code', flags=re.IGNORECASE)
unnecessary_code_cols = [c for c in cols if code_regex.search(c)]

# 3) Print results for review
print("ID-like columns to review (excluding 'SINH_ID'):")
for c in unnecessary_id_cols:
    print("-", c)
print(f"Total ID-like columns: {len(unnecessary_id_cols)}\n")

print("CODE-like columns to review:")
for c in unnecessary_code_cols:
    print("-", c)
print(f"Total CODE-like columns: {len(unnecessary_code_cols)}")


In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Drop columns listed in a user-defined mapping from `data`
# into a NEW DataFrame (`data_no_ids_codes`). 
# -----------------------------------------------------------------------------
# Instructions:
# 1) Define `cols_to_remove_map` yourself (e.g., {"colA": "id", "colB": "code"}).
# 2) This cell will:
#       • print the mapping,
#       • create `data_backup` (full copy),
#       • create `data_no_ids_codes` with those columns removed,
#       • report shapes.
# 3) It does NOT overwrite `data`. To adopt later:  data = data_no_ids_codes
# -----------------------------------------------------------------------------

# 1) Define your mapping here (column_name -> reason string like 'id' or 'code')
# 1) Define your list of columns to drop
cols_to_remove = [
    "sdo1_prev_distance_id",
    "sdo1_postal_distance_id",
    "sdo5_degree_degree_code_letters",
    "sdo5_degree_degree_code",
    "sdo5_degree_POSTCODE"
]

# 2) Print for confirmation
print("Columns to drop:")
for col in cols_to_remove:
    print("-", col)
print(f"\nTotal columns to dropped: {len(cols_to_remove)}")

# 3) Create backup and drop
data_backup = data.copy()
data_no_ids_codes = data.drop(columns=cols_to_remove, errors='ignore')

print("\nOriginal shape:", data.shape, "-> New shape:", data_no_ids_codes.shape)

# Optional adoption:
# data = data_no_ids_codes



In [ ]:
data = data_no_ids_codes

In [ ]:
# Ensure no two or more columns have the same values
identical_columns = []
columns = data.columns.tolist()

for i in range(len(columns)):
    for j in range(i+1, len(columns)):
        col1, col2 = columns[i], columns[j]
        if data[col1].equals(data[col2]):
            identical_columns.append((col1, col2))

if identical_columns:
    print("Identical columns found:")
    for col_pair in identical_columns:
        print(col_pair)
else:
    print("No identical columns found.")

In [ ]:
# Check for columns with only one unique value
single_value_cols = [col for col in data.columns if data[col].nunique() == 1]

print("Columns with only one unique value:", single_value_cols)


In [ ]:
# print the value 
print(data["has_results"].value_counts(dropna=False))

In [ ]:
# has_results has only one value, delete it

del data['has_results']

In [ ]:
data.info()

In [ ]:
#pd.set_option('display.max_rows', None)   # Show all rows (be careful if huge!)
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_colwidth', None)
#pd.set_option('display.width', 0)         # Automatically adjusts line-wrapping
#pd.set_option('display.expand_frame_repr', False)  


In [ ]:
# Print value counts for each column
for col in data.columns:
    print(f"Value counts for column: {col}")
    print(data[col].value_counts(dropna=False))  # include NaN values in the count
    print("\n" + "="*50 + "\n")

In [ ]:
data.columns

In [ ]:
# Redundant columns: keep sdo1_characteristics_Dutch_nationality (yes/no 1/0), delete sdo5_degree_POSTAL_COUNTRY_NL

del data['sdo5_degree_POSTAL_COUNTRY_NL']

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Identify columns whose names contain 'date' or 'datum' (any case),
# print them, then convert those columns to pandas datetime.
# Notes:
# - Uses case-insensitive search for 'date' or 'datum'.
# - Converts IN PLACE on `data` using pd.to_datetime(errors='coerce', dayfirst=True).
#   (dayfirst=True is common for NL-style dates; adjust if needed.)
# -----------------------------------------------------------------------------

# 1) Find date-related columns
date_pattern = re.compile(r"(date|datum)", flags=re.IGNORECASE)
date_cols = [c for c in data.columns if date_pattern.search(c)]

# 2) Print for review
print("Detected date-related columns:")
for c in date_cols:
    print("-", c)
print(f"\nNumber of converted columns to date/time: {len(date_cols)}")

# 3) Convert to datetime (in place)
for c in date_cols:
    # Normalize obvious empties to NaN before parsing
    if data[c].dtype == object:
        data[c] = data[c].replace({"": np.nan, " ": np.nan, "NA": np.nan, "N/A": np.nan})
    data[c] = pd.to_datetime(data[c], errors="coerce", dayfirst=True)


In [ ]:
# Print value counts only for object (categorical) columns, excluding datetime columns and ID columns
# Columns to skip manually
skip_cols = ["SINH_ID"]

for col in data.select_dtypes(include=["object"]).columns:
    if col in skip_cols or col in data.select_dtypes(include=["datetime"]).columns:
        continue
    print(f"Value counts for column: {col}")
    print(data[col].value_counts(dropna=False))  # include NaN values in the count
    print("\n" + "="*50 + "\n")


In [ ]:
# -----------------------------------------------------------------------------
# Purpose: For the next step, Print the object (categorical) columns that will be used
# in the next value-counts loop — excluding ID and datetime columns.
# -----------------------------------------------------------------------------

skip_cols = ["SINH_ID"]

dt_cols = set(data.select_dtypes(include=["datetime64[ns]"]).columns)
obj_cols_all = list(data.select_dtypes(include=["object"]).columns)
obj_cols = [c for c in obj_cols_all if c not in dt_cols and c not in skip_cols]

print("Object dtype columns (excluding datetime and skipped IDs):\n")
for col in obj_cols:
    print(col)
print(f"\nTotal object columns: {len(obj_cols)}")


----------------------- Handle categories in sdo5_degree_degree Column ---------------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Clean and standardize `sdo5_degree_degree` program names for analysis.
# What this cell does:
#   • Applies exact mappings for specific categories:
#       - "B Education in Primary Schools (age 4 - 12) (day)"     → "Primary_Education_Day"
#       - "B Education in Primary Schools (age 4 - 12) (ALPO)"    → "Primary_Education_ALPO"
#       - "B Education in Primary Schools (age 4 - 12) (Regular)" → "Primary_Education_Regular"
#       - "B Arts Therapies (Music Therapy)"                      → "Music_Therapy"
#       - "B Arts Therapies (Drama Therapy)"                      → "Drama_Therapy"
#       - "B Arts Therapies (Art Therapy)"                        → "Art_Therapy"
#   • Removes leading degree prefixes: "B " and "Bachelor of ".
#   • Fixes known typo: "Chemics" → "Chemistry".
#   • Removes prepositions "and", "with", "in" (case-insensitive), commas, and ampersands.
#   • Normalizes whitespace, converts to Title Case, then replaces spaces with underscores.
#   • Preserves the standalone category "Teacher" exactly as-is.
#   • Converts NaN values to the label "Unknown".
# -----------------------------------------------------------------------------


col = "sdo5_degree_degree"

def clean_degree(value):
    if pd.isna(value):
        return "Unknown"
    
    v = str(value).strip()

    # ---- Exact mappings to keep as specified ----
    exact_map = {
        "B Education in Primary Schools (age 4 - 12) (day)": "Primary_Education_Day",
        "B Education in Primary Schools (age 4 - 12) (ALPO)": "Primary_Education_ALPO",
        "B Education in Primary Schools (age 4 - 12) (Regular)": "Primary_Education_Regular",
        "B Arts Therapies (Music Therapy)": "Music_Therapy",
        "B Arts Therapies (Drama Therapy)": "Drama_Therapy",
        "B Arts Therapies (Art Therapy)": "Art_Therapy",
        # Optional: make "Bachelor of Law" explicit instead of "Law"
        # "Bachelor of Law": "Law_Bachelor",
    }
    if v in exact_map:
        return exact_map[v]

    # ---- Remove prefixes ----
    v = re.sub(r"^\s*B\s+", "", v)
    v = re.sub(r"^\s*Bachelor of\s+", "", v, flags=re.IGNORECASE)

    # ---- Fix known inconsistencies ----
    v = v.replace("Chemics", "Chemistry")

    # ---- Remove prepositions and punctuation ----
    v = v.replace("&", " ")
    v = re.sub(r"\b(?:and|with|in)\b", "", v, flags=re.IGNORECASE)
    v = v.replace(",", " ")

    # ---- Normalize whitespace and casing ----
    v = re.sub(r"\s+", " ", v).strip()
    v = v.title()

    # ---- Replace spaces with underscores ----
    v = v.replace(" ", "_")

    # ---- NEW: collapse multiple underscores and trim edges ----
    v = re.sub(r"_+", "_", v).strip("_")

    return v

# Apply cleaning directly to the original column
data[col] = data[col].apply(clean_degree)

# Optional check
print(data[col].value_counts())

----------------------- Handle categories in sdo1_characteristics_gender Column ---------------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: In `sdo1_characteristics_gender`, replace NaN and 'O'/'o' with 'Unknown'
# without altering other values (e.g., 'M', 'V').
# -----------------------------------------------------------------------------

col = "sdo1_characteristics_gender"

# If the column is categorical, ensure 'Unknown' exists as a category
import pandas as pd
if pd.api.types.is_categorical_dtype(data[col]):
    data[col] = data[col].cat.add_categories(["Unknown"])

# Replace 'O'/'o' and fill NaN
data[col] = data[col].replace({"O": "Unknown", "o": "Unknown"}).fillna("Unknown")
print(data['sdo1_characteristics_gender'].value_counts())


------------------- Handle categories in sdo1_previous_Previous_Education_Type Column ---------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: In `sdo1_previous_Previous_Education_Type`, replace NaN and literal
# '$' with 'Unknown
# -----------------------------------------------------------------------------


col = "sdo1_previous_Previous_Education_Type"

# If categorical, ensure 'Unknown' is an allowed category
if pd.api.types.is_categorical_dtype(data[col]):
    data[col] = data[col].cat.add_categories(["Unknown"])

# Replace and fill
data[col] = data[col].replace({"$": "Unknown"}).fillna("Unknown")

# Print value counts after replacement
print(data[col].value_counts())


----------------- Handle categories in sdo1_previous_Previous_School_Postal_Code Column -------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Replace NaN (missing) values in
# 'sdo1_previous_Previous_School_Postal_Code' with 'Unknown'.
# -----------------------------------------------------------------------------

col = "sdo1_previous_Previous_School_Postal_Code"

# If the column is categorical, ensure 'Unknown' exists as a category
if pd.api.types.is_categorical_dtype(data[col]):
    data[col] = data[col].cat.add_categories(["Unknown"])

# Fill missing values with 'Unknown'
data[col] = data[col].fillna("Unknown")
print(data[col].value_counts())

-------------------------------------- Handle categories in sdo2_ADVIES_DEF Column -------------------------------------------

In [ ]:
# -----------------------------------------------------------------------------
# Purpose: Replace NaN with 'Unknown' and add underscores between words
# in the categories of `sdo2_skc_ADVIES_DEF.
# -----------------------------------------------------------------------------

col = "sdo2_skc_ADVIES_DEF"

# 1) Add underscores between words for non-missing values
data[col] = data[col].apply(lambda x: "_".join(str(x).split()) if pd.notna(x) else x)

# 2) Replace NaN with 'Unknown'
data[col] = data[col].fillna("Unknown")
print(data[col].value_counts())

In [ ]:
out_path = OUT_DIR / "cleaned_original.csv"
data.to_csv(out_path, index=False)